Explore **every USGS-recorded mine, adit, shaft, and prospect** in Nye County, NV and all its neighbors — right from your browser.

**How to use this:** click **Runtime ▸ Run all** (Colab) or the ⏩ button (Jupyter). Wait ~1 minute for data to download. Then scroll to the **Playground** section and change the settings to ask your own questions.

In [1]:
# @title 1️⃣ Setup (installs — takes ~30s, ignore the wall of text)
%pip -q install pandas duckdb folium requests
print('✅ ready')


Note: you may need to restart the kernel to use updated packages.
✅ ready


In [2]:
# @title 2️⃣ Download USGS data (cached — only downloads the first time)
import os, io, zipfile, requests, pandas as pd

os.makedirs('data', exist_ok=True)

def fetch(url, name):
    path = f'data/{name}'
    if not os.path.exists(path):
        print(f'downloading {name} ...')
        r = requests.get(url, timeout=300); r.raise_for_status()
        open(path, 'wb').write(r.content)
    return path

# MRDS: worldwide mineral-site database (26 MB zip)
z = zipfile.ZipFile(fetch('https://mrdata.usgs.gov/mrds/mrds-csv.zip', 'mrds-csv.zip'))
if not os.path.exists('data/mrds.csv'):
    z.extract('mrds.csv', 'data')

# USMIN: mine features digitized from topo maps (points only, via dbf+shp)
for st in ['NV', 'CA']:
    zipfile.ZipFile(fetch(f'https://mrdata.usgs.gov/usmin/state/usmin-{st}.zip', f'usmin-{st}.zip')).extractall(f'data/usmin-{st}')
print('✅ data on disk:', sorted(os.listdir('data')))


downloading mrds-csv.zip ...
downloading usmin-NV.zip ...
downloading usmin-CA.zip ...
✅ data on disk: ['mrds-csv.zip', 'mrds.csv', 'usmin-CA', 'usmin-CA.zip', 'usmin-NV', 'usmin-NV.zip']


In [3]:
# @title 3️⃣ Load + filter to the Tonopah region
import struct, math

NV_COUNTIES = ['Nye','Esmeralda','Mineral','Churchill','Lander','Eureka','White Pine','Lincoln','Clark']
CA_COUNTIES = ['Inyo']   # add 'Mono' if you like

# --- MRDS sites (rich attributes: commodities, status, geology) ---
mrds_all = pd.read_csv('data/mrds.csv', low_memory=False)
mrds = mrds_all[((mrds_all.state=='Nevada') & (mrds_all.county.isin(NV_COUNTIES))) |
                ((mrds_all.state=='California') & (mrds_all.county.isin(CA_COUNTIES)))].copy()

# --- USMIN features: read shapefiles with a tiny pure-python reader (no GDAL needed) ---
def read_points_shp(base):
    # .shp for coords (point type only), .dbf for attributes
    with open(base + '.shp','rb') as f:
        buf = f.read()
    pts, off = [], 100
    while off < len(buf):
        _, clen = struct.unpack('>ii', buf[off:off+8])
        shp_type = struct.unpack('<i', buf[off+8:off+12])[0]
        if shp_type == 1:
            x, y = struct.unpack('<dd', buf[off+12:off+28])
            pts.append((x, y))
        else:
            pts.append((None, None))
        off += 8 + clen*2
    with open(base + '.dbf','rb') as f:
        d = f.read()
    n_rec, hdr, rlen = struct.unpack('<I', d[4:8])[0], struct.unpack('<H', d[8:10])[0], struct.unpack('<H', d[10:12])[0]
    fields, p = [], 32
    while d[p] != 0x0d:
        name = d[p:p+11].split(b'\x00')[0].decode()
        fields.append((name, d[p+16]))
        p += 32
    rows = []
    for i in range(n_rec):
        rec, q, row = d[hdr+i*rlen:hdr+(i+1)*rlen], 1, {}
        for name, ln in fields:
            row[name] = rec[q:q+ln].decode('latin-1').strip(); q += ln
        rows.append(row)
    out = pd.DataFrame(rows)
    out['lon'] = [p[0] for p in pts][:len(out)]
    out['lat'] = [p[1] for p in pts][:len(out)]
    return out

usmin = pd.concat([
    read_points_shp('data/usmin-NV/NV-point').query('COUNTY in @NV_COUNTIES'),
    read_points_shp('data/usmin-CA/CA-point').query('COUNTY in @CA_COUNTIES'),
], ignore_index=True)

# distance from Tonopah for every feature
def miles_from_tonopah(lat, lon, LAT0=38.0672, LON0=-117.2301):
    R=3958.76; p1,p2=math.radians(LAT0),math.radians(lat)
    dp,dl=math.radians(lat-LAT0),math.radians(lon-LON0)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))
usmin['miles_from_tonopah'] = [round(miles_from_tonopah(la,lo),1) if la else None for la,lo in zip(usmin.lat, usmin.lon)]
mrds['miles_from_tonopah'] = [round(miles_from_tonopah(la,lo),1) if pd.notna(la) else None for la,lo in zip(mrds.latitude, mrds.longitude)]

print(f'✅ {len(mrds):,} MRDS sites | {len(usmin):,} USMIN mapped features')


✅ 11,398 MRDS sites | 97,139 USMIN mapped features


## Datasets

- **USMIN** = *where the holes in the ground are* (adits, shafts, pits — traced off old topo maps)
- **MRDS** = *what was mined and how it panned out* (commodities, producer vs. prospect, geology)


In [4]:
print('USMIN — physical features by type:')
display(usmin.FTR_TYPE.value_counts().head(12).to_frame('count'))
print('MRDS — sites by development status:')
display(mrds.dev_stat.value_counts().to_frame('count'))
print('MRDS — top primary commodities in the region:')
display(mrds.commod1.value_counts().head(15).to_frame('count'))


USMIN — physical features by type:


,count
FTR_TYPE,
Prospect Pit,69212
Adit,12508
Mine Shaft,11752
Gravel Pit,1326
Open Pit Mine,952
Borrow Pit,839
Quarry,212
Open Pit Mine or Quarry,153
Gravel/Borrow Pit - Undifferentiated,65


MRDS — sites by development status:


,count
dev_stat,
Past Producer,4946
Occurrence,2911
Prospect,2063
Unknown,697
Producer,615
Plant,166


MRDS — top primary commodities in the region:


,count
commod1,
Gold,2961
Silver,1208
Tungsten,612
Copper,591
Lead,523
"Gold, Silver",435
Mercury,301
"Silver, Gold",294
Uranium,272


---
## Playground

Change any value below, then re-run this cell and the two after it (Shift+Enter runs a cell).


In [5]:
# ============== CHANGE ME ==============
FEATURE_TYPES = ['Adit', 'Mine Shaft']   # try: 'Prospect Pit', 'Open Pit Mine', 'Quarry', 'Tunnel'
MAX_MILES     = 50                        # radius from Tonopah (blank = no limit: None)
COUNTIES      = None                      # None = all, or e.g. ['Nye', 'Esmeralda']
NAMED_ONLY    = False                     # True = only features with a name
# =======================================

q = usmin[usmin.FTR_TYPE.isin(FEATURE_TYPES)]
if MAX_MILES:  q = q[q.miles_from_tonopah <= MAX_MILES]
if COUNTIES:   q = q[q.COUNTY.isin(COUNTIES)]
if NAMED_ONLY: q = q[q.FTR_NAME != '']
q = q.sort_values('miles_from_tonopah')

print(f' {len(q):,} features match')
display(q[['FTR_TYPE','FTR_NAME','COUNTY','STATE','miles_from_tonopah','TOPO_NAME']].head(25))


 5,441 features match


,FTR_TYPE,FTR_NAME,COUNTY,STATE,miles_from_tonopah,TOPO_NAME
75553,Mine Shaft,,Nye,NV,0.1,Tonopah
13655,Mine Shaft,,Nye,NV,0.1,Tonopah
75233,Mine Shaft,,Nye,NV,0.1,Tonopah
14155,Mine Shaft,,Nye,NV,0.1,Tonopah
75174,Mine Shaft,,Nye,NV,0.2,Tonopah
75507,Mine Shaft,,Nye,NV,0.2,Tonopah
14921,Mine Shaft,,Nye,NV,0.2,Tonopah
13823,Mine Shaft,,Nye,NV,0.2,Tonopah
75352,Mine Shaft,,Nye,NV,0.2,Tonopah
13954,Mine Shaft,,Nye,NV,0.2,Tonopah


In [6]:
# @title 🗺️ Map the matches (auto-capped at 2,000 markers)
import folium
from folium.plugins import MarkerCluster

m = folium.Map(location=[38.0672,-117.2301], zoom_start=8, tiles='OpenStreetMap')
folium.Marker([38.0672,-117.2301], tooltip='Tonopah', icon=folium.Icon(color='red', icon='star')).add_to(m)
cluster = MarkerCluster().add_to(m)
for _, r in q.head(2000).iterrows():
    folium.CircleMarker([r.lat, r.lon], radius=4, fill=True,
        popup=f"{r.FTR_NAME or '(unnamed)'}<br>{r.FTR_TYPE} — {r.COUNTY} Co.<br>{r.miles_from_tonopah} mi from Tonopah"
    ).add_to(cluster)
m


## MRDS: what was mined around here?


In [7]:
# ============== CHANGE ME ==============
COMMODITY  = 'Gold'      # try: 'Silver', 'Copper', 'Turquoise', 'Uranium', 'Lithium', 'Mercury'
STATUS     = 'Producer'  # try: 'Past Producer', 'Prospect', 'Occurrence', or None for all
MILES      = 60
# =======================================

hits = mrds[(mrds[['commod1','commod2','commod3']].apply(lambda c: c.str.contains(COMMODITY, case=False, na=False)).any(axis=1))]
if STATUS: hits = hits[hits.dev_stat.str.contains(STATUS, na=False)]
if MILES:  hits = hits[hits.miles_from_tonopah <= MILES]
hits = hits.sort_values('miles_from_tonopah')
print(f' {len(hits):,} MRDS sites match')
display(hits[['site_name','commod1','commod2','dev_stat','work_type','county','miles_from_tonopah','url']].head(25))


 728 MRDS sites match


,site_name,commod1,commod2,dev_stat,work_type,county,miles_from_tonopah,url
288278,West End Consolidated,Silver,NaN,Past Producer,NaN,Nye,0.2,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...
39433,West End Consolid. Mining Co.,"Silver, Gold",NaN,Past Producer,NaN,Nye,0.2,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...
193005,Tonopah Midway Mine,Silver,NaN,Past Producer,NaN,Nye,0.3,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...
43233,Tonopah Mining Co.,"Silver, Gold",NaN,Producer,Underground,Nye,0.3,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...
169046,Mcnamara,Gold,NaN,Past Producer,NaN,Nye,0.3,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...
43226,Montana - Tonopah Mining Co.,Silver,Gold,Past Producer,NaN,Nye,0.4,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...
287848,Montana Tonopah,Silver,NaN,Past Producer,NaN,Nye,0.4,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...
264479,Gypsy Queen,Silver,NaN,Past Producer,NaN,Nye,0.5,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...
169349,Victor,Silver,NaN,Past Producer,NaN,Nye,0.5,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...
122010,Sand Grass,Silver,NaN,Past Producer,NaN,Nye,0.5,https://mrdata.usgs.gov/mrds/show-mrds.php?dep...


## SQL query

The same dataframes are queryable with plain SQL via DuckDB:


In [8]:
import duckdb
duckdb.sql('''
  SELECT county, count(*) AS sites,
         sum(CASE WHEN dev_stat LIKE '%Producer%' THEN 1 ELSE 0 END) AS producers
  FROM mrds
  GROUP BY county ORDER BY sites DESC
''').df()


,county,sites,producers
0,Nye,2042,876.0
1,Inyo,1945,848.0
2,Esmeralda,1365,554.0
3,Lander,1138,768.0
4,Mineral,1115,626.0
5,Lincoln,840,361.0
6,White Pine,833,447.0
7,Clark,825,454.0
8,Churchill,786,339.0
9,Eureka,509,288.0
